In [24]:
import os

from pathlib import Path

import matplotlib.pyplot as plt
import torch
import hashlib

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN, QuantumCircuitGraphDataset
import torch.nn as nn
from torch_geometric.loader import DataLoader

In [25]:
model_global = torch.load("../models/gnn_model_global.pt", map_location=torch.device('cpu'), weights_only=True)
model_cliff = torch.load("../models/gnn_model_clifford.pt", map_location=torch.device('cpu'), weights_only=True)
model_haar = torch.load("../models/gnn_model_haar.pt", map_location=torch.device('cpu'), weights_only=True)
model_quan = torch.load("../models/gnn_model_quansistor.pt", map_location=torch.device('cpu'), weights_only=True)
model_rand = torch.load("../models/gnn_model_random.pt", map_location=torch.device('cpu'), weights_only=True)

In [26]:
model_global

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0799,  0.1218,  0.1017,  ...,  0.0616,  0.3570, -0.1041],
                       [ 0.0666, -0.1135,  0.2196,  ..., -0.1784,  0.1784,  0.0697],
                       [ 0.0504, -0.0783,  0.1665,  ..., -0.2898,  0.1836,  0.1277],
                       ...,
                       [-0.0908,  0.1287,  0.0495,  ...,  0.0377, -0.0582,  0.1709],
                       [ 0.0582,  0.1228,  0.0250,  ..., -0.0553, -0.0971, -0.1235],
                       [-0.1111, -0.0992,  0.0718,  ..., -0.0090,  0.1854, -0.2600]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 3.3768e-02,  3.0306e-02, -1.3022e-01,  1.1655e-01, -9.4946e-02,
                        8.5620e-03,  4.8090e-02, -7.0045e-02, -4.0032e-02,  9.6853e-02,
                       -8.8972e-02,  6.5760e-02, -3.4950e-02, -5.7792e-02,  1.1348e-01,
                        4.0260e-02,  7.6376e-02,  1.0220e-01, -7.1336e-02, -1.0660e

In [27]:
model_cliff

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.0172, -0.1633,  0.0092,  ...,  0.1165, -0.0710, -0.1990],
                       [-0.1047,  0.0422,  0.3499,  ...,  0.1588, -0.1267,  0.0111],
                       [-0.1595,  0.0175,  0.3035,  ..., -0.1548, -0.2756, -0.0727],
                       ...,
                       [ 0.0920,  0.1971,  0.0036,  ..., -0.1476, -0.1123,  0.2426],
                       [ 0.0875,  0.2268, -0.1706,  ..., -0.1004, -0.1500, -0.0891],
                       [ 0.2065, -0.0319,  0.1331,  ...,  0.0269, -0.1515, -0.1303]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.2139,  0.1764, -0.0217,  0.1926, -0.0402,  0.1581, -0.1077, -0.1067,
                       -0.1644, -0.2116,  0.0946, -0.0823,  0.0272, -0.1482, -0.1449, -0.0005,
                        0.1807,  0.0983,  0.2336,  0.1154, -0.0183,  0.0504, -0.0915,  0.0549,
                        0.1466,  0.1443,  0.1790,  0.0363,  0.

In [28]:
model_quan

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-0.0605,  0.1926, -0.0793,  ...,  0.0554,  0.0169,  0.1751],
                       [ 0.0862, -0.1949, -0.0061,  ...,  0.2289,  0.2139, -0.0582],
                       [-0.1668, -0.0457, -0.0174,  ...,  0.0817, -0.1172,  0.2682],
                       ...,
                       [ 0.0675,  0.0920, -0.0662,  ..., -0.1882, -0.0514,  0.0242],
                       [-0.1136,  0.0980, -0.2644,  ...,  0.2751, -0.3158,  0.1560],
                       [ 0.0431,  0.2636, -0.0647,  ...,  0.0040,  0.1140,  0.2429]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.1048,  0.2508,  0.2164,  0.2142, -0.2119, -0.1251, -0.2042,  0.1413,
                       -0.1945, -0.1697,  0.1407,  0.2600,  0.2412, -0.2098,  0.2014,  0.1729,
                        0.0134, -0.0315,  0.1755,  0.1511, -0.2541,  0.1761, -0.0309, -0.1918,
                       -0.0065, -0.1839,  0.2449,  0.0938,  0.

In [29]:
model_rand

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[-0.0437, -0.1226, -0.1669,  ...,  0.3157, -0.1526,  0.2178],
                       [-0.1908, -0.0545,  0.1828,  ..., -0.0327,  0.1367,  0.2220],
                       [ 0.2097,  0.0056, -0.1930,  ..., -0.1858, -0.3114, -0.1222],
                       ...,
                       [ 0.0349,  0.0886,  0.2126,  ...,  0.0270, -0.1124,  0.0342],
                       [ 0.0286, -0.0895, -0.1739,  ..., -0.0992, -0.2846, -0.1087],
                       [-0.0765,  0.1601,  0.0335,  ..., -0.1048, -0.1140,  0.1287]])),
              ('conv_layers.0.lin_key.bias',
               tensor([ 0.0249, -0.1824, -0.0844,  0.0757,  0.1287,  0.0006, -0.2018,  0.1251,
                        0.0909, -0.1803,  0.1736,  0.0677,  0.0259, -0.2374, -0.1966,  0.1270,
                       -0.1608, -0.0429,  0.0288,  0.0243,  0.1164,  0.0885, -0.1910, -0.2017,
                       -0.2391, -0.0405,  0.1397, -0.2158,  0.

In [30]:
model_haar

{'model_state_dict': OrderedDict([('conv_layers.0.lin_key.weight',
               tensor([[ 0.1258,  0.0976, -0.1237,  ...,  0.1709, -0.0154,  0.1079],
                       [ 0.0573,  0.1374, -0.0513,  ..., -0.0012,  0.0217, -0.1830],
                       [-0.0292,  0.0787, -0.1406,  ..., -0.0115, -0.0545,  0.2044],
                       ...,
                       [-0.0128,  0.1076, -0.1050,  ..., -0.0556, -0.0044,  0.1336],
                       [ 0.0345,  0.0254, -0.0964,  ..., -0.1252,  0.0918,  0.0036],
                       [ 0.0341, -0.0564,  0.0491,  ..., -0.0612, -0.0358, -0.1106]])),
              ('conv_layers.0.lin_key.bias',
               tensor([-0.1375,  0.0806, -0.1294, -0.0370,  0.0175, -0.0411, -0.1156, -0.0296,
                        0.1250,  0.1450, -0.0364, -0.0702, -0.0018,  0.1057,  0.0316,  0.1392,
                       -0.0950,  0.1428, -0.1232,  0.0033,  0.0423,  0.0161, -0.1087,  0.0152,
                        0.0947,  0.0085,  0.0226, -0.0199,  0.

In [31]:
def collect_pred_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    pred_root = d / "predictions"

    if family is not None:
        paths = sorted((pred_root / family).glob("*.pt"))
    else:
        paths = []
        if pred_root.exists():
            for family_dir in sorted(pred_root.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))

    if not paths:
        paths = sorted(d.glob("*.pt"))

    return [str(p.resolve()) for p in paths]


def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    # Include full resolved paths in the digest to avoid collisions across folders/families.
    canonical = "|".join(sorted(str(Path(p).resolve()) for p in paths))
    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]
    tag = f"_{suffix}" if suffix else ""
    cache_dir = Path("..") / "qqe" / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    return str(cache_dir.resolve())

In [32]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

In [33]:
def build_pred_loaders_two_stage(
    pt_paths: list[str],
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[QuantumCircuitGraphDataset, PaddedGraphDatasetWrapper,DataLoader, int, int]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )

    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    padded_dataset = PaddedGraphDatasetWrapper(dataset)
    node_in_dim = padded_dataset.target_dim
    global_in_dim = dataset.global_feature_dim

    pred_ds = padded_dataset
    pin_mem = torch.cuda.is_available()
    return (
        dataset,
        padded_dataset,
        DataLoader(pred_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_mem),
        node_in_dim,
        global_in_dim,
    )

In [34]:
family = "haar"
global_feature_variant = "binned"
node_feature_backend_variant = None

In [35]:
MODEL_STATE_PATH = f"../models/gnn_model_{family}.pt"
CHECKPOINT_PATH = f"../models/gnn_model_{family}.pt"

In [36]:
pred_data_paths = collect_pred_paths("../outputs/data/predictions", family=family)
print(f"Collected {len(pred_data_paths)} .pt files for dataset.")

Collected 9000 .pt files for dataset.


In [37]:
dataset, padded_data, pred_loader, node_in_dim, global_in_dim = build_pred_loaders_two_stage(
    pred_data_paths,  # Use first 10 paths for demonstration
    batch_size=32,
)

In [38]:
dataset[0]

Data(
  x=[70, 55],
  edge_index=[2, 110],
  y=[1],
  global_features=[1, 3],
  num_qubits=10,
  gate_counts={ haar_count=50 },
  meta={
    cid='haar_Q10_L11_S1041247627',
    family='haar',
    n_qubits=10,
    n_layers=11,
    seed=1041247627,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [39]:
# --------------------------------------------------
# 1. Build prediction dataset with training-compatible feature config
# --------------------------------------------------
if not pred_data_paths:
    raise RuntimeError("No prediction .pt files found. Check outputs/data/predictions/<family>.")

checkpoint = None
model_config = {}
fixed_all_gate_keys = None

if Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    if isinstance(checkpoint, dict):
        model_config = checkpoint.get("model_config", {}) or {}
        feature_config = checkpoint.get("feature_config", {}) or {}
        fixed_all_gate_keys = feature_config.get("all_gate_keys", None)

suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
root = _cache_root_for_paths(pred_data_paths, suffix=suffix)

pred_dataset = QuantumCircuitGraphDataset(
    root=root,
    pt_paths=pred_data_paths,
    global_feature_variant=global_feature_variant,
    node_feature_backend_variant=node_feature_backend_variant,
    fixed_all_gate_keys=fixed_all_gate_keys,
)

trained_node_in_dim = model_config.get("node_in_dim", None)
trained_global_in_dim = model_config.get("global_in_dim", None)

padded_pred_dataset = PaddedGraphDatasetWrapper(
    pred_dataset,
    target_node_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
    target_global_dim=trained_global_in_dim if trained_global_in_dim is not None else None,
    target_dim=trained_node_in_dim if trained_node_in_dim is not None else None,
 )

pred_loader = DataLoader(
    padded_pred_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

node_in_dim = padded_pred_dataset.target_dim
global_in_dim = padded_pred_dataset.target_global_dim

print(f"Prediction graphs: {len(pred_dataset)}")
print(f"Prediction node feature dim: {node_in_dim}")
print(f"Prediction global feature dim: {global_in_dim}")
if trained_node_in_dim is not None or trained_global_in_dim is not None:
    print(f"Trained node/global dims from checkpoint: {trained_node_in_dim}/{trained_global_in_dim}")

C:\Users\Victor\AppData\Local\Temp\ipykernel_336336\351848649.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu

Prediction graphs: 9000
Prediction node feature dim: 45
Prediction global feature dim: 3
Trained node/global dims from checkpoint: 45/3


In [40]:
pred_dataset[0]

Data(
  x=[70, 55],
  edge_index=[2, 110],
  y=[1],
  global_features=[1, 3],
  num_qubits=10,
  gate_counts={ haar_count=50 },
  meta={
    cid='haar_Q10_L11_S1041247627',
    family='haar',
    n_qubits=10,
    n_layers=11,
    seed=1041247627,
    backend='pennylane',
    method='fwht',
    representation='dense',
    n_bins=50,
  }
)

In [41]:
# --------------------------------------------------
# 2. Validate dataset/model dimensions
# --------------------------------------------------
print("node_in_dim (prediction) =", node_in_dim)
print("global_in_dim (prediction) =", global_in_dim)
print("node_in_dim (trained) =", trained_node_in_dim)
print("global_in_dim (trained) =", trained_global_in_dim)

if trained_node_in_dim is not None and node_in_dim != trained_node_in_dim:
    print(f"Node dim will be adapted to trained dim {trained_node_in_dim} at inference time.")

if trained_global_in_dim is not None and global_in_dim != trained_global_in_dim:
    print(f"Global feature dim will be adapted to trained dim {trained_global_in_dim} at inference time.")

node_in_dim (prediction) = 45
global_in_dim (prediction) = 3
node_in_dim (trained) = 45
global_in_dim (trained) = 3


In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [43]:
# --------------------------------------------------
# 3. Rebuild model and load weights robustly
# --------------------------------------------------
def _extract_state_dict(payload):
    if isinstance(payload, nn.Module):
        return payload.state_dict()
    if isinstance(payload, dict) and "model_state_dict" in payload:
        return payload["model_state_dict"]
    if isinstance(payload, dict) and all(torch.is_tensor(v) for v in payload.values()):
        return payload
    raise RuntimeError("Unsupported model file format.")

if checkpoint is not None and isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    if not Path(MODEL_STATE_PATH).exists():
        raise FileNotFoundError(f"Could not find model weights at {MODEL_STATE_PATH}")
    raw_payload = torch.load(MODEL_STATE_PATH, map_location="cpu")
    state_dict = _extract_state_dict(raw_payload)

model_kwargs = {
    "node_in_dim": int(trained_node_in_dim if trained_node_in_dim is not None else node_in_dim),
    "global_in_dim": int(trained_global_in_dim if trained_global_in_dim is not None else global_in_dim),
    "gnn_hidden": int(model_config.get("gnn_hidden", 32)),
    "gnn_heads": int(model_config.get("gnn_heads", 8)),
    "global_hidden": int(model_config.get("global_hidden", 16)),
    "reg_hidden": int(model_config.get("reg_hidden", 16)),
    "num_layers": int(model_config.get("num_layers", 5)),
    "dropout_rate": float(model_config.get("dropout_rate", 0.1)),
}

model = GNN(**model_kwargs).to(device)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
model.eval()

print("Loaded model config:", model_kwargs)
if missing_keys:
    print("Missing keys:", missing_keys)
if unexpected_keys:
    print("Unexpected keys:", unexpected_keys)

Loaded model config: {'node_in_dim': 45, 'global_in_dim': 3, 'gnn_hidden': 32, 'gnn_heads': 8, 'global_hidden': 16, 'reg_hidden': 16, 'num_layers': 5, 'dropout_rate': 0.1}


In [44]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [11.040547370910645, 11.138599395751953, 10.938201904296875, 11.108991622924805, 11.113648414611816, 11.197930335998535, 10.866480827331543, 10.970351219177246, 10.966170310974121, 10.952306747436523]
Total predictions: 9000


In [45]:
# --------------------------------------------------
# 4. Predict with defensive dimension adaptation
# --------------------------------------------------
predictions = []
expected_node_dim = model_kwargs["node_in_dim"]
expected_global_dim = model_kwargs["global_in_dim"]

with torch.no_grad():
    for batch in pred_loader:
        # Adapt node features to the model input dimension.
        if batch.x.size(1) < expected_node_dim:
            pad_size = expected_node_dim - batch.x.size(1)
            batch.x = torch.nn.functional.pad(batch.x, (0, pad_size), mode="constant", value=0.0)
        elif batch.x.size(1) > expected_node_dim:
            batch.x = batch.x[:, :expected_node_dim]

        # Adapt global features to the model input dimension.
        g = batch.global_features
        if g.dim() == 1:
            g = g.view(batch.num_graphs, -1)
        if g.size(1) < expected_global_dim:
            g = torch.nn.functional.pad(g, (0, expected_global_dim - g.size(1)), mode="constant", value=0.0)
        elif g.size(1) > expected_global_dim:
            g = g[:, :expected_global_dim]
        batch.global_features = g

        batch = batch.to(device)
        pred = model(batch).view(-1)
        predictions.extend(pred.cpu().tolist())

print("First 10 predictions:", predictions[:10])
print("Total predictions:", len(predictions))

First 10 predictions: [11.040547370910645, 11.138599395751953, 10.938201904296875, 11.108990669250488, 11.113648414611816, 11.197929382324219, 10.86648178100586, 10.97035026550293, 10.966172218322754, 10.952305793762207]
Total predictions: 9000


In [48]:
min(predictions)

4.416338920593262

In [ ]:
def collect_pt_paths(dataset_dir: str, family: str | None = None) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    if family is not None:
        paths = sorted((d / "encoding_data_pennylane" / family).glob("*.pt"))
    else:
        paths = []
        encoding_dir = d / "encoding_data_pennylane"
        if encoding_dir.exists():
            for family_dir in sorted(encoding_dir.iterdir()):
                if family_dir.is_dir():
                    paths.extend(sorted(family_dir.glob("*.pt")))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

: 

: 

In [ ]:
class PaddedGraphDatasetWrapper:
    """Wrapper that pads/truncates graph features to consistent dimensions."""

    def __init__(
        self,
        dataset,
        target_node_dim: int | None = None,
        target_global_dim: int | None = None,
        target_dim: int | None = None,  # backwards compatibility
    ):
        self.dataset = dataset

        # Backwards compatibility with older call sites using target_dim.
        if target_node_dim is None and target_dim is not None:
            target_node_dim = target_dim

        self.target_dim = target_node_dim if target_node_dim is not None else self._compute_max_node_dim()
        self.target_global_dim = (
            target_global_dim if target_global_dim is not None else self._compute_max_global_dim()
        )

    def _compute_max_node_dim(self) -> int:
        """Find max node feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            x = getattr(data, "x", None)
            if x is not None and x.dim() > 1:
                max_dim = max(max_dim, int(x.shape[1]))
        return max_dim

    def _compute_max_global_dim(self) -> int:
        """Find max global feature width across all samples."""
        max_dim = 0
        for i in range(len(self.dataset)):
            data = self.dataset[i]
            g = getattr(data, "global_features", None)
            if g is None:
                continue
            if g.dim() == 0:
                width = 1
            elif g.dim() == 1:
                width = int(g.shape[0])
            else:
                width = int(g.shape[-1])
            max_dim = max(max_dim, width)
        return max_dim

    def _fit_node_dim(self, data):
        x = getattr(data, "x", None)
        if x is None or x.dim() <= 1:
            return data
        current = int(x.shape[1])
        if current == self.target_dim:
            return data
        out = data.clone()
        if current < self.target_dim:
            pad_size = self.target_dim - current
            out.x = torch.nn.functional.pad(out.x, (0, pad_size), mode="constant", value=0.0)
        else:
            out.x = out.x[:, : self.target_dim]
        return out

    def _fit_global_dim(self, data):
        g = getattr(data, "global_features", None)
        if g is None or self.target_global_dim <= 0:
            return data

        out = data.clone() if out_is_same(data, g) else data
        g = getattr(out, "global_features")

        # Ensure graph-level vector shape.
        if g.dim() == 0:
            g = g.view(1)
        elif g.dim() > 1:
            g = g.view(-1)

        current = int(g.shape[0])
        if current < self.target_global_dim:
            g = torch.nn.functional.pad(
                g, (0, self.target_global_dim - current), mode="constant", value=0.0
            )
        elif current > self.target_global_dim:
            g = g[: self.target_global_dim]

        out.global_features = g
        return out

    def __getitem__(self, idx: int):
        data = self.dataset[idx]
        data = self._fit_node_dim(data)
        data = self._fit_global_dim(data)
        return data

    def __len__(self) -> int:
        return len(self.dataset)


def out_is_same(data, g):
    # Clone lazily only when we actually need to edit global features.
    return hasattr(data, "global_features") and data.global_features is g

: 

: 

In [ ]:
data_paths = collect_pt_paths("../outputs/data", family=None)
print(f"Collected {len(data_paths)} .pt files for dataset.")

Collected 40000 .pt files for dataset.


: 

: 

: 

: 